# Serve Base Gemma 4 E2B + Tuned Planner/Refiner V3 Endpoints

This Colab notebook starts one Flask app behind ngrok with four routes:

- `/planner/generate` -> base Gemma 4 E2B planner behavior, no adapter
- `/refiner/generate` -> base Gemma 4 E2B refiner behavior, no adapter
- `/planner-v3/generate` -> tuned Planner V3 LoRA adapter from `planner_v3_e2b_unsloth_qlora_grounded_full_v1.zip`
- `/refiner-v3-preset/generate` -> tuned Refiner V3 preset LoRA adapter from `refiner_v3_preset_e2b_unsloth_qlora_v1.zip`

Use the tuned Planner V3 endpoint when testing the VASP v3 media-transcript matching planner, and use the tuned Refiner V3 preset endpoint when testing professional preset-based styling.

Note: If dependency installation upgrades `pyarrow` or `numpy`, restart the Colab runtime once before loading Unsloth. Colab can keep old binary extensions in memory otherwise.


In [1]:
# 1) Check GPU runtime
!nvidia-smi


Sat May 16 22:34:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 2) Install dependencies
# IMPORTANT: this cell intentionally restarts the Colab runtime after pip changes.
# Do not continue to the model-loading cell in the same runtime after installing.
# Reason: Unsloth checks loaded numpy vs installed numpy. If pip upgrades numpy
# while the kernel still has the old C-extension loaded, Unsloth raises:
#   RuntimeError: numpy was upgraded mid-session ... restart your runtime/kernel

import os, sys

!pip -q install -U --no-cache-dir unsloth peft transformers accelerate bitsandbytes flask pyngrok requests huggingface_hub
!pip -q uninstall -y pyarrow pyarrow-hotfix datasets >/dev/null 2>&1
!pip -q install --no-cache-dir --force-reinstall "pyarrow==17.0.0" "datasets==3.6.0"

print('Dependencies installed. Colab runtime will restart now.')
print('After it restarts, rerun cells from the top, but skip this install cell unless you need to reinstall.')
# os.kill(os.getpid(), 9)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 144.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 417.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 208.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 735.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 935.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 506.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 403.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 465.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 432.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 386.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 801.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
# 3) Hugging Face login (needed if the Gemma checkpoint is gated)
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
# 4) Mount Drive and unzip tuned Planner V3 + Refiner V3 preset adapters
from pathlib import Path
import shutil

from google.colab import drive
drive.mount('/content/drive')

PLANNER_V3_ZIP = Path('/content/drive/MyDrive/planner_v3_e2b_unsloth_qlora_grounded_full_v1.zip')
REFINER_V3_PRESET_ZIP = Path('/content/drive/MyDrive/refiner_v3_preset_e2b_unsloth_qlora_v1.zip')
PLANNER_V3_UNZIP_ROOT = Path('/content/adapters/planner_v3')
REFINER_V3_PRESET_UNZIP_ROOT = Path('/content/adapters/refiner_v3_preset')

if not PLANNER_V3_ZIP.exists():
    raise FileNotFoundError(
        f'Tuned planner adapter zip not found: {PLANNER_V3_ZIP}\n'
        'Upload planner_v3_e2b_unsloth_qlora_grounded_full_v1.zip to your Google Drive MyDrive folder, '
        'or change PLANNER_V3_ZIP in this cell.'
    )

if not REFINER_V3_PRESET_ZIP.exists():
    raise FileNotFoundError(
        f'Tuned refiner preset adapter zip not found: {REFINER_V3_PRESET_ZIP}\n'
        'Upload refiner_v3_preset_e2b_unsloth_qlora_v1.zip to your Google Drive MyDrive folder, '
        'or change REFINER_V3_PRESET_ZIP in this cell.'
    )

for root in (PLANNER_V3_UNZIP_ROOT, REFINER_V3_PRESET_UNZIP_ROOT):
    if root.exists():
        shutil.rmtree(root)
    root.mkdir(parents=True, exist_ok=True)

!unzip -q -o "{PLANNER_V3_ZIP}" -d "{PLANNER_V3_UNZIP_ROOT}"
!unzip -q -o "{REFINER_V3_PRESET_ZIP}" -d "{REFINER_V3_PRESET_UNZIP_ROOT}"

def find_adapter_dir(root: Path, zip_path: Path) -> Path:
    candidates = sorted(root.rglob('adapter_config.json'))
    if not candidates:
        raise FileNotFoundError(f'No adapter_config.json found after unzipping {zip_path}')

    # Prefer the actual adapter weight directory if the zip contains nested folders.
    for cfg in candidates:
        parent = cfg.parent
        if (parent / 'adapter_model.safetensors').exists() or (parent / 'adapter_model.bin').exists():
            return parent
    return candidates[0].parent

planner_v3_adapter_dir = find_adapter_dir(PLANNER_V3_UNZIP_ROOT, PLANNER_V3_ZIP)
refiner_v3_preset_adapter_dir = find_adapter_dir(REFINER_V3_PRESET_UNZIP_ROOT, REFINER_V3_PRESET_ZIP)

print('Planner V3 adapter dir:', planner_v3_adapter_dir)
print('Planner adapter files:', sorted(x.name for x in planner_v3_adapter_dir.iterdir())[:20])
print('\nRefiner V3 preset adapter dir:', refiner_v3_preset_adapter_dir)
print('Refiner preset adapter files:', sorted(x.name for x in refiner_v3_preset_adapter_dir.iterdir())[:20])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Planner V3 adapter dir: /content/adapters/planner_v3/vasp/a2v/finetuning/planner_v3_dataset/output/planner_v3_e2b_unsloth_qlora_grounded_full_v1/adapter/planner_v3
Planner adapter files: ['adapter_config.json', 'adapter_model.safetensors']

Refiner V3 preset adapter dir: /content/adapters/refiner_v3_preset/vasp/a2v/finetuning/refiner_v3_presets/output/refiner_v3_preset_e2b_unsloth_qlora_v1/adapter
Refiner preset adapter files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json']


In [4]:
# 5) Load base Gemma 4 E2B with Unsloth and attach tuned Planner V3 + Refiner V3 preset adapters
# If this cell says numpy was upgraded mid-session, restart runtime and rerun from the top.
import importlib.metadata as importlib_metadata
import sys

try:
    import numpy as _np
    _loaded_numpy = _np.__version__
    _installed_numpy = importlib_metadata.version('numpy')
    if _loaded_numpy != _installed_numpy:
        raise RuntimeError(
            f'numpy mismatch before Unsloth import: loaded={_loaded_numpy}, installed={_installed_numpy}. '
            'Restart the Colab runtime, then rerun from cell 1 and skip the install cell.'
        )
except Exception as exc:
    if 'numpy mismatch' in str(exc):
        raise
    print('numpy preflight skipped:', exc)

import torch
from unsloth import FastLanguageModel
from peft import PeftModel

BASE_MODEL = 'google/gemma-4-e2b-it'
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

if getattr(tokenizer, 'pad_token', None) is None and hasattr(tokenizer, 'eos_token'):
    tokenizer.pad_token = tokenizer.eos_token

if hasattr(tokenizer, 'padding_side'):
    tokenizer.padding_side = 'left'


def unwrap_gemma4_clippable_linear(module, prefix=''):
    """PEFT cannot inject LoRA into Gemma4ClippableLinear wrappers.

    The trained adapter targets names like q_proj/k_proj/v_proj, but in Gemma 4
    those modules can be wrapper objects containing an inner .linear layer.
    Replacing the wrapper with its inner torch.nn.Linear keeps the same module
    name, so adapter keys still match while PEFT sees a supported target type.
    """
    replaced = []
    for name, child in list(module.named_children()):
        child_path = f'{prefix}.{name}' if prefix else name
        if child.__class__.__name__ == 'Gemma4ClippableLinear' and hasattr(child, 'linear'):
            setattr(module, name, child.linear)
            replaced.append(child_path)
            continue
        replaced.extend(unwrap_gemma4_clippable_linear(child, child_path))
    return replaced

unwrapped_modules = unwrap_gemma4_clippable_linear(model)
print(f'Unwrapped Gemma4ClippableLinear modules for PEFT: {len(unwrapped_modules)}')
if unwrapped_modules:
    print('First unwrapped modules:', unwrapped_modules[:12])

# Attach tuned Planner V3 adapter first.
model = PeftModel.from_pretrained(
    model,
    str(planner_v3_adapter_dir),
    adapter_name='planner_v3',
    is_trainable=False,
)

# Attach tuned Refiner V3 preset adapter as a second LoRA adapter.
model.load_adapter(
    str(refiner_v3_preset_adapter_dir),
    adapter_name='refiner_v3_preset',
    is_trainable=False,
)

model.eval()
FastLanguageModel.for_inference(model)

print('Loaded base model:', BASE_MODEL)
print('Loaded adapters:', list(getattr(model, 'peft_config', {}).keys()))
print('Base planner/refiner routes will disable adapters; /planner-v3/generate enables planner_v3; /refiner-v3-preset/generate enables refiner_v3_preset.')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Unwrapped Gemma4ClippableLinear modules for PEFT: 232
First unwrapped modules: ['model.vision_tower.encoder.layers.0.self_attn.q_proj', 'model.vision_tower.encoder.layers.0.self_attn.k_proj', 'model.vision_tower.encoder.layers.0.self_attn.v_proj', 'model.vision_tower.encoder.layers.0.self_attn.o_proj', 'model.vision_tower.encoder.layers.0.mlp.gate_proj', 'model.vision_tower.encoder.layers.0.mlp.up_proj', 'model.vision_tower.encoder.layers.0.mlp.down_proj', 'model.vision_tower.encoder.layers.1.self_attn.q_proj', 'model.vision_tower.encoder.layers.1.self_attn.k_proj', 'model.vision_tower.encoder.layers.1.self_attn.v_proj', 'model.vision_tower.encoder.layers.1.self_attn.o_proj', 'model.vision_tower.encoder.layers.1.mlp.gate_proj']
Loaded base model: google/gemma-4-e2b-it
Loaded adapters: ['planner_v3', 'refiner_v3_preset']
Base planner/refiner routes will disable adapters; /planner-v3/generate enables planner_v3; /refiner-v3-preset/generate enables refiner_v3_preset.


In [5]:
# Optional: free the serving port if an older Flask process is still running.
!for pid in $(lsof -t -i:8089 2>/dev/null); do kill -9 $pid; done || true


In [6]:
# 6) Clean old ngrok tunnels / local Flask processes if needed
!pkill -f "python.*flask" || true
!pkill -f "python.*8097" || true

from pyngrok import ngrok
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass
ngrok.kill()


^C
^C


ERROR:pyngrok.process.ngrok:t=2026-05-16T22:36:16+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-16T22:36:16+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-16T22:36:16+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

In [7]:
# 7) Start Flask app + one ngrok tunnel with base planner/refiner + tuned Planner V3 + tuned Refiner V3 preset routes
import json
import re
import subprocess
import threading
import time
import traceback
import requests
from threading import Lock

import torch
from flask import Flask, jsonify, request
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

NGROK_AUTHTOKEN = NGROK

ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Make this cell safe to rerun. If the same static/free ngrok domain is already
# online from this runtime, this usually clears it. If it is online in another
# Colab runtime, ngrok will still reject and we print a clear next step below.
try:
    for tunnel in ngrok.get_tunnels():
        try:
            ngrok.disconnect(tunnel.public_url)
            print('Disconnected old tunnel:', tunnel.public_url)
        except Exception as exc:
            print('Could not disconnect tunnel:', tunnel.public_url, exc)
except Exception:
    pass
try:
    ngrok.kill()
except Exception:
    pass
subprocess.run(['pkill', '-f', 'ngrok'], check=False)
time.sleep(2)

SERVER_PORT = 8089
app = Flask('base_e2b_plus_tuned_planner_refiner_v3')
_gen_lock = Lock()


def _payload_value(payload, key, default):
    value = payload.get(key, default)
    return default if value is None else value


def _generate_text(prompt: str, max_new_tokens: int = 1200, temperature: float = 0.0) -> str:
    messages = [{'role': 'user', 'content': prompt}]
    rendered_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    if rendered_prompt is None:
        # Some Gemma 4 processor builds return None for tokenize=False. Keep a simple Gemma chat fallback.
        rendered_prompt = f'<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n'

    # Gemma 4 may expose a Processor, where positional args are interpreted as images.
    # Always pass text= explicitly so the prompt does not become the images argument.
    inputs = tokenizer(text=rendered_prompt, return_tensors='pt')
    inputs = inputs.to(model.device) if hasattr(inputs, 'to') else {k: v.to(model.device) for k, v in inputs.items()}
    input_len = int(inputs['input_ids'].shape[-1])

    eos_token_id = getattr(tokenizer, 'eos_token_id', None)
    if eos_token_id is None and hasattr(tokenizer, 'tokenizer'):
        eos_token_id = tokenizer.tokenizer.eos_token_id

    gen_kwargs = {
        'max_new_tokens': int(max_new_tokens),
        'pad_token_id': eos_token_id,
        'eos_token_id': eos_token_id,
    }
    if float(temperature) > 0:
        gen_kwargs.update({'do_sample': True, 'temperature': float(temperature), 'top_p': 0.9})
    else:
        gen_kwargs.update({'do_sample': False})

    with torch.inference_mode():
        output_ids = model.generate(**inputs, **gen_kwargs)

    decoder = getattr(tokenizer, 'tokenizer', tokenizer)
    return decoder.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()


def _generate_base(prompt: str, max_new_tokens: int, temperature: float) -> str:
    # PEFT exposes disable_adapter(), which lets the existing base planner/refiner routes remain pure base model.
    with _gen_lock:
        if hasattr(model, 'disable_adapter'):
            with model.disable_adapter():
                return _generate_text(prompt, max_new_tokens=max_new_tokens, temperature=temperature)
        return _generate_text(prompt, max_new_tokens=max_new_tokens, temperature=temperature)


def _clean_planner_v3_response(text: str) -> str:
    raw = (text or '').strip()
    try:
        obj = json.loads(raw)
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        pass

    marker = '{\"planner_version\"'
    starts = [m.start() for m in re.finditer(re.escape(marker), raw)]
    for start in reversed(starts):
        candidate = raw[start:].strip()
        for suffix in ('', '}', ', \"warnings\": []}'):
            try:
                obj = json.loads(candidate + suffix)
                if isinstance(obj, dict) and obj.get('planner_version') == 'v3_media_text_matching':
                    obj.setdefault('warnings', [])
                    return json.dumps(obj, ensure_ascii=False)
            except Exception:
                continue
    return raw


def _generate_planner_v3(prompt: str, max_new_tokens: int, temperature: float) -> str:
    with _gen_lock:
        model.set_adapter('planner_v3')
        response = _generate_text(prompt, max_new_tokens=max_new_tokens, temperature=temperature)
        return _clean_planner_v3_response(response)


def _generate_refiner_v3_preset(prompt: str, max_new_tokens: int, temperature: float) -> str:
    with _gen_lock:
        model.set_adapter('refiner_v3_preset')
        return _generate_text(prompt, max_new_tokens=max_new_tokens, temperature=temperature)


def _route_generate(mode: str):
    payload = request.get_json(force=True, silent=False) or {}
    prompt = _payload_value(payload, 'prompt', '')
    if not prompt:
        return jsonify({'error': 'prompt is required'}), 400

    max_tokens = int(_payload_value(payload, 'max_tokens',4000))
    temperature = float(_payload_value(payload, 'temperature', 0.0))

    try:
        if mode == 'planner_v3':
            response = _generate_planner_v3(prompt, max_tokens, temperature)
        elif mode == 'refiner_v3_preset':
            response = _generate_refiner_v3_preset(prompt, max_tokens, temperature)
        else:
            response = _generate_base(prompt, max_tokens, temperature)
        return jsonify({'response': response, 'mode': mode, 'model': BASE_MODEL})
    except Exception as exc:
        tb = traceback.format_exc()
        print(f'[{mode}] generation failed:', exc)
        print(tb)
        return jsonify({'error': str(exc), 'traceback': tb, 'mode': mode}), 500


@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'ok': True,
        'model': BASE_MODEL,
        'base_routes': ['/planner/generate', '/refiner/generate'],
        'adapter_routes': ['/planner-v3/generate', '/refiner-v3-preset/generate'],
        'adapters': list(getattr(model, 'peft_config', {}).keys()),
    })


@app.route('/planner/generate', methods=['POST'])
def planner_generate():
    return _route_generate('base_planner')


@app.route('/refiner/generate', methods=['POST'])
def refiner_generate():
    return _route_generate('base_refiner')


@app.route('/planner-v3/generate', methods=['POST'])
def planner_v3_generate():
    return _route_generate('planner_v3')


@app.route('/refiner-v3-preset/generate', methods=['POST'])
def refiner_v3_preset_generate():
    return _route_generate('refiner_v3_preset')


def run_app():
    app.run(host='0.0.0.0', port=SERVER_PORT, debug=False, use_reloader=False)

thread = threading.Thread(target=run_app, daemon=True)
thread.start()

# Wait until Flask is actually serving before creating the ngrok tunnel.
local_health_url = f'http://127.0.0.1:{SERVER_PORT}/health'
last_health_error = None
for _ in range(30):
    try:
        r = requests.get(local_health_url, timeout=1)
        if r.status_code == 200:
            print('Local Flask health OK:', local_health_url, r.json())
            break
    except Exception as exc:
        last_health_error = exc
    time.sleep(1)
else:
    raise RuntimeError(
        f'Flask did not start on localhost:{SERVER_PORT}. Last health error: {last_health_error}. '
        'Check the cell output above for Flask startup errors or port conflicts.'
    )

try:
    public_url = ngrok.connect(SERVER_PORT, bind_tls=True).public_url
except PyngrokNgrokHTTPError as exc:
    msg = str(exc)
    if 'ERR_NGROK_334' in msg or 'already online' in msg:
        raise RuntimeError(
            'ngrok says this domain is already online. Stop the old notebook/runtime that is using it, '
            'or open https://dashboard.ngrok.com/endpoints and stop the active endpoint, then rerun this cell. '
            'If you are intentionally running multiple notebooks, use a different ngrok account/token.'
        ) from exc
    raise

planner_url = public_url + '/planner/generate'
refiner_url = public_url + '/refiner/generate'
planner_v3_url = public_url + '/planner-v3/generate'
refiner_v3_preset_url = public_url + '/refiner-v3-preset/generate'

print('PUBLIC URL:', public_url)
print('Base planner endpoint:', planner_url)
print('Base refiner endpoint:', refiner_url)
print('Tuned Planner V3 endpoint:', planner_v3_url)
print('Tuned Refiner V3 preset endpoint:', refiner_v3_preset_url)
print('Health:', public_url + '/health')


 * Serving Flask app 'base_e2b_plus_tuned_planner_refiner_v3'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8089
 * Running on http://172.28.0.12:8089
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:36:20] "GET /health HTTP/1.1" 200 -


Local Flask health OK: http://127.0.0.1:8089/health {'adapter_routes': ['/planner-v3/generate', '/refiner-v3-preset/generate'], 'adapters': ['planner_v3', 'refiner_v3_preset'], 'base_routes': ['/planner/generate', '/refiner/generate'], 'model': 'google/gemma-4-e2b-it', 'ok': True}
PUBLIC URL: https://bungee-swell-smashing.ngrok-free.dev
Base planner endpoint: https://bungee-swell-smashing.ngrok-free.dev/planner/generate
Base refiner endpoint: https://bungee-swell-smashing.ngrok-free.dev/refiner/generate
Tuned Planner V3 endpoint: https://bungee-swell-smashing.ngrok-free.dev/planner-v3/generate
Tuned Refiner V3 preset endpoint: https://bungee-swell-smashing.ngrok-free.dev/refiner-v3-preset/generate
Health: https://bungee-swell-smashing.ngrok-free.dev/health


In [8]:
# 8) Smoke test endpoints
import requests

sample_prompt = 'Return valid JSON only: {"ok": true, "route": "test"}'

for name, url in [
    ('base planner', planner_url),
    ('base refiner', refiner_url),
    ('tuned planner v3', planner_v3_url),
    ('tuned refiner v3 preset', refiner_v3_preset_url),
]:
    r = requests.post(url, json={'prompt': sample_prompt, 'temperature': 0.0, 'max_tokens': 80}, timeout=120)
    print('\n---', name, r.status_code, '---')
    print(r.text[:1000])


INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:36:36] "POST /planner/generate HTTP/1.1" 200 -



--- base planner 200 ---
{"mode":"base_planner","model":"google/gemma-4-e2b-it","response":"```json\n{\"ok\": true, \"route\": \"test\"}\n```"}



INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:36:37] "POST /refiner/generate HTTP/1.1" 200 -



--- base refiner 200 ---
{"mode":"base_refiner","model":"google/gemma-4-e2b-it","response":"```json\n{\"ok\": true, \"route\": \"test\"}\n```"}



INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:36:42] "POST /planner-v3/generate HTTP/1.1" 200 -



--- tuned planner v3 200 ---
{"mode":"planner_v3","model":"google/gemma-4-e2b-it","response":"{\"ok\": true, \"route\": \"test\"}"}



INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:36:47] "POST /refiner-v3-preset/generate HTTP/1.1" 200 -



--- tuned refiner v3 preset 200 ---
{"mode":"refiner_v3_preset","model":"google/gemma-4-e2b-it","response":"```json\n{\"ok\": true, \"route\": \"test\"}\n```"}



INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:39:42] "POST /planner-v3/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:40:20] "POST /refiner-v3-preset/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:40:42] "POST /refiner-v3-preset/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:41:10] "POST /refiner-v3-preset/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:41:38] "POST /refiner-v3-preset/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:42:22] "POST /refiner-v3-preset/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:42:55] "POST /refiner-v3-preset/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:43:23] "POST /refiner-v3-preset/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:44:11] "POST /refiner-v3-preset/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 22:44:54] "POST /refiner-v3-preset/generate HTTP/1.1" 200

In [ ]:
# 9) Use these URLs in VASP CLI
print('Base planner endpoint:')
print(planner_url)
print('\nTuned Planner V3 endpoint:')
print(planner_v3_url)
print('\nBase refiner endpoint:')
print(refiner_url)
print('\nTuned Refiner V3 preset endpoint:')
print(refiner_v3_preset_url)

print('\nFor VASP v3 testing, use tuned planner + tuned preset refiner:')
print(f'--planner-endpoint "{planner_v3_url}"')
print(f'--refiner-endpoint "{refiner_v3_preset_url}"')

print('\nExample:')
print('python -m vasp.a2v.v3.new_flow_pipeline_v3 `')
print('  --edit-name "edit2" `')
print('  --captions-file "assets/inputs/edit2/captions.txt" `')
print('  --instruction "Create a clean engaging short-form video with synced captions." `')
print(f'  --planner-endpoint "{planner_v3_url}" `')
print(f'  --refiner-endpoint "{refiner_v3_preset_url}" `')
print('  --creativity 4')


Base planner endpoint:
https://bungee-swell-smashing.ngrok-free.dev/planner/generate

Tuned Planner V3 endpoint:
https://bungee-swell-smashing.ngrok-free.dev/planner-v3/generate

Base refiner endpoint:
https://bungee-swell-smashing.ngrok-free.dev/refiner/generate

For VASP v3 testing, use tuned planner + base refiner:
--planner-endpoint "https://bungee-swell-smashing.ngrok-free.dev/planner-v3/generate"
--refiner-endpoint "https://bungee-swell-smashing.ngrok-free.dev/refiner/generate"

Example:
python -m vasp.a2v.v3.new_flow_pipeline_v3 `
  --edit-name "edit2" `
  --captions-file "assets/inputs/edit2/captions.txt" `
  --instruction "Create a clean engaging short-form video with synced captions." `
  --planner-endpoint "https://bungee-swell-smashing.ngrok-free.dev/planner-v3/generate" `
  --refiner-endpoint "https://bungee-swell-smashing.ngrok-free.dev/refiner/generate"


INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:22:25] "POST /planner-v3/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:22:50] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:23:23] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:23:56] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:24:29] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:25:09] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:25:41] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:26:14] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:26:53] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:27:18] "POST /refiner/generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/May/2026 16:28:07] "POST /refiner/generate HTTP/1.1" 20

: 